# Part 5: Model Evaluation and Improvement

**Course:** 2026 KMITL Data Analytics

This notebook shows how to measure a model honestly: the confusion matrix and
classification metrics, balanced and imbalanced data, training/validation/test
splits, K-fold cross-validation, underfitting and overfitting, bias and
variance, L1 and L2 regularization, hyperparameter tuning, data leakage and
`Pipeline`.


## Learning objectives

By the end of this notebook you can:

1. Read a binary confusion matrix with true positives, true negatives, false positives and false negatives.
2. Explain the positive-class convention and why it matters.
3. Compute accuracy, precision, recall, F1, Jaccard and log loss from a small example.
4. Compare a majority-class dummy model with a trained model on balanced and imbalanced data.
5. Split data into training, validation and test parts and describe generalization.
6. Run K-fold cross-validation with `StratifiedKFold` and `cross_val_score`.
7. Put preprocessing and a model in a `Pipeline` so it is fitted inside each fold, avoiding data leakage.
8. Explain underfitting, overfitting, bias and variance.
9. Compare models with and without L1 and L2 regularization.
10. Separate hyperparameters from learned parameters and tune them with `GridSearchCV` on training data only.


## Required imports

Colab usually includes these libraries; for local use, install the project
requirements first. We use scikit-learn for data, models and metrics, and
Matplotlib and Seaborn for charts. `%matplotlib inline` shows charts inside the
page.


In [ ]:
import matplotlib

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
    log_loss,
    mean_squared_error,
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded.")


## Dataset introduction

This notebook needs no downloads. It builds five small datasets in memory:

| Dataset | Size | Purpose |
|---|---|---|
| Hand example | 10 predictions | Read the confusion matrix and every metric by hand. |
| Balanced classification | 400 rows, two equal classes | Compare a dummy model and a trained model fairly. |
| Imbalanced classification | 400 rows, 95% / 5% | Show why accuracy alone can mislead. |
| Synthetic regression curve | 60 points | Show underfitting, overfitting and regularization. |
| Tuning dataset | 300 rows | Tune a hyperparameter without touching the test set. |

The **positive class** is label `1`. It stands for the event we care about, such
as "has the condition" or "is fraud". The negative class is label `0`. All data
is synthetic and only for teaching.


In [ ]:
# The hand example: ten predictions. Positive class = 1.
actual = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
predicted = np.array([1, 1, 1, 1, 0, 1, 0, 0, 0, 0])

hand_table = pd.DataFrame({"actual": actual, "predicted": predicted})
hand_table.index = range(1, 11)
print(hand_table)


## Confusion matrix

A **confusion matrix** counts how often each class is predicted. For binary
classification it has four cells. With the **positive class** fixed as label `1`:

- **True positive (TP)**: actual `1`, predicted `1`. Correct alarm.
- **True negative (TN)**: actual `0`, predicted `0`. Correct all-clear.
- **False positive (FP)**: actual `0`, predicted `1`. False alarm.
- **False negative (FN)**: actual `1`, predicted `0`. Missed case.

scikit-learn prints the matrix in this fixed layout:

```text
                  Predicted 0     Predicted 1
Actual 0             TN              FP
Actual 1             FN              TP
```

Always state the positive class. Swapping it swaps FP and FN, which changes
precision and recall.


### Small numerical example

Count the ten rows of the hand table.

| # | actual | predicted | cell |
|---|---|---|---|
| 1 | 1 | 1 | TP |
| 2 | 1 | 1 | TP |
| 3 | 1 | 1 | TP |
| 4 | 1 | 1 | TP |
| 5 | 1 | 0 | FN |
| 6 | 0 | 1 | FP |
| 7 | 0 | 0 | TN |
| 8 | 0 | 0 | TN |
| 9 | 0 | 0 | TN |
| 10 | 0 | 0 | TN |

Counts: `TP = 4`, `FN = 1`, `FP = 1`, `TN = 4`. Check the total:
`TP + FN + FP + TN = 4 + 1 + 1 + 4 = 10`, the number of rows.


In [ ]:
# scikit-learn returns the matrix in this order: [[TN, FP], [FN, TP]].
matrix = confusion_matrix(actual, predicted, labels=[0, 1])
tn, fp, fn, tp = matrix.ravel()
print("confusion matrix [[TN, FP], [FN, TP]]:")
print(matrix)
print("TN =", tn, "FP =", fp, "FN =", fn, "TP =", tp)

# Check against the hand counts.
assert (tn, fp, fn, tp) == (4, 1, 1, 4)

fig, ax = plt.subplots(figsize=(5, 4))
display = ConfusionMatrixDisplay(
    confusion_matrix=matrix, display_labels=["Negative (0)", "Positive (1)"]
)
display.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix of the Hand Example")
plt.tight_layout()
plt.show()


**Reading the matrix.** The top-left `4` is TN, the top-right `1` is FP, the
bottom-left `1` is FN and the bottom-right `4` is TP. The model misses one
positive case (FN) and raises one false alarm (FP). Most cells are on the
diagonal (TN and TP), so the model is right for 8 of the 10 rows.


## Classification metrics

Each metric answers a different question. We compute every one on the same four
counts: `TP = 4`, `FP = 1`, `FN = 1`, `TN = 4`.

**Accuracy** asks: what share of all predictions is correct?
Hand example: `(TP + TN) / total = (4 + 4) / 10 = 0.8`.
General form:

```text
accuracy = (TP + TN) / (TP + TN + FP + FN)
```

**Precision** asks: when we predict positive, how often are we right?
Hand example: `TP / (TP + FP) = 4 / (4 + 1) = 0.8`.
General form:

```text
precision = TP / (TP + FP)
```

**Recall** (sensitivity) asks: of the real positive cases, how many did we find?
Hand example: `TP / (TP + FN) = 4 / (4 + 1) = 0.8`.
General form:

```text
recall = TP / (TP + FN)
```

**F1-score** balances precision and recall in one number. It is the harmonic
mean, so it is low if either one is low.
Hand example: `2 * 0.8 * 0.8 / (0.8 + 0.8) = 1.28 / 1.6 = 0.8`.
General form:

```text
F1 = 2 * precision * recall / (precision + recall)
```

**Jaccard score** (intersection over union) compares the predicted positive set
with the actual positive set.
Hand example: `TP / (TP + FP + FN) = 4 / (4 + 1 + 1) = 4 / 6 = 0.667`.
General form:

```text
Jaccard = TP / (TP + FP + FN)
```

**Zero division.** If `TP + FP = 0`, precision is `0 / 0`. scikit-learn does not
guess; it returns `0` when we pass `zero_division=0` (or `1` to treat it as
perfect). Choose one on purpose and say which.


In [ ]:
accuracy = accuracy_score(actual, predicted)
precision = precision_score(actual, predicted, pos_label=1, zero_division=0)
recall = recall_score(actual, predicted, pos_label=1, zero_division=0)
f1 = f1_score(actual, predicted, pos_label=1, zero_division=0)
jaccard = jaccard_score(actual, predicted, pos_label=1, zero_division=0)

metric_table = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall", "F1", "Jaccard"],
    "value": [accuracy, precision, recall, f1, jaccard],
})
print(metric_table.round(3))

# Check against the hand arithmetic.
assert np.isclose(accuracy, 0.8)
assert np.isclose(precision, 0.8)
assert np.isclose(recall, 0.8)
assert np.isclose(f1, 0.8)
assert np.isclose(jaccard, 4 / 6)
print("All classification metrics match the hand example.")


**Reading the metric table.** All four counts give accuracy, precision, recall and
F1 equal to `0.8`, while Jaccard is about `0.667`. F1 and Jaccard both use TP, FP
and FN, but F1 weights precision and recall equally, so it reads higher than
Jaccard here. A high accuracy does not tell you which kind of error the model
makes; the confusion matrix and the other metrics do.


## Log loss

**Log loss** (cross-entropy) scores predicted **probabilities**, not hard
labels. A model that says `0.9` for a true positive is good; a model that says
`0.01` for a true positive is punished hard. The natural logarithm `ln` makes the
punishment grow without limit as the probability moves toward the wrong answer.

### Small numerical example

Two predictions:

- Example A: actual `1`, predicted probability `0.9`.
  Loss = `-ln(0.9) = 0.105`.
- Example B: actual `0`, predicted probability `0.8`.
  Loss = `-ln(1 - 0.8) = -ln(0.2) = 1.609`.

Average: `(0.105 + 1.609) / 2 = 0.857`.

General form, for `n` examples with actual label `y_i` and predicted probability
`p_i`:

```text
log_loss = -(1 / n) * sum of [ y_i * ln(p_i) + (1 - y_i) * ln(1 - p_i) ]
```

Here `y_i` is `0` or `1`, `p_i` is the probability of class `1`, and the sum runs
over all examples. Only one term of each pair is non-zero.

**Zero division.** `ln(0)` is minus infinity. scikit-learn clips probabilities
into a tiny range such as `[1e-15, 1 - 1e-15]` so the loss stays finite; it never
returns infinite loss for a real model.


In [ ]:
# Tiny hand example: two predictions with probabilities.
tiny_actual = np.array([1, 0])
tiny_probs = np.array([0.9, 0.8])
tiny_log_loss = log_loss(tiny_actual, tiny_probs, labels=[0, 1])
print("tiny log loss:", round(tiny_log_loss, 3))
assert np.isclose(tiny_log_loss, 0.8574, atol=1e-3)

# The same 10 examples as before, now with probabilities instead of labels.
probs = np.array([0.9, 0.8, 0.7, 0.6, 0.45, 0.55, 0.2, 0.1, 0.25, 0.15])
example_log_loss = log_loss(actual, probs, labels=[0, 1])
print("log loss of the 10 examples:", round(example_log_loss, 3))

# A confident wrong prediction has a large loss.
confident_wrong = log_loss(np.array([1]), np.array([0.01]), labels=[0, 1])
print("loss of a confident wrong prediction:", round(confident_wrong, 3))
assert confident_wrong > tiny_log_loss
print("Log loss checks passed.")


**Reading the log-loss output.** The two-example value is about `0.857`, matching
the hand arithmetic. The ten-example value is about `0.357` because those
probabilities are mostly confident and correct. The confident wrong prediction
has a much larger loss, which shows why log loss cares about probabilities and
not only about the final label.


## Balanced and imbalanced datasets

A **balanced** dataset has classes of similar size. An **imbalanced** dataset has
one class much rarer than the other, for example 95% negative and 5% positive.
Many real problems are imbalanced: fraud, disease and machine failure are rare.

On imbalanced data, a model that always predicts the majority class can have high
**accuracy** and still be useless, because it never finds the rare positive
class. Its precision, recall and F1 are `0`. This is why we compare accuracy with
F1.

**Small numerical example.** In 100 rows, 95 are negative and 5 positive. A model
that always says "negative" is right `95 / 100 = 0.95` of the time, but it finds
`0` of the 5 positive cases. Accuracy `0.95`, recall `0`, F1 `0`.


In [ ]:
# Balanced: the two classes are about equally common.
X_balanced, y_balanced = make_classification(
    n_samples=400,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.5, 0.5],
    class_sep=1.5,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

# Imbalanced: the positive class (1) is rare.
X_imbalanced, y_imbalanced = make_classification(
    n_samples=400,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.95, 0.05],
    class_sep=2.0,
    flip_y=0.01,
    random_state=RANDOM_STATE,
)

print("balanced class counts:", np.bincount(y_balanced))
print("imbalanced class counts:", np.bincount(y_imbalanced))


In [ ]:
# One honest split per dataset, then compare the same two models.
def compare_dummy_and_logistic(X, y, dataset_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
    )

    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train, y_train)
    dummy_pred = dummy.predict(X_test)

    logistic = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    logistic.fit(X_train, y_train)
    logistic_pred = logistic.predict(X_test)

    rows = []
    for name, pred in [("Dummy (majority)", dummy_pred), ("LogisticRegression", logistic_pred)]:
        rows.append({
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "F1": f1_score(y_test, pred, pos_label=1, zero_division=0),
        })

    result = pd.DataFrame(rows)
    result.insert(0, "dataset", dataset_name)
    # The split is disjoint: the two parts add up to the whole.
    assert len(X_train) + len(X_test) == len(X)
    return result


balanced_result = compare_dummy_and_logistic(X_balanced, y_balanced, "balanced")
imbalanced_result = compare_dummy_and_logistic(X_imbalanced, y_imbalanced, "imbalanced")
comparison = pd.concat([balanced_result, imbalanced_result], ignore_index=True)
print(comparison.round(3))

# The majority dummy never predicts the positive class, so its F1 is 0.
assert comparison.loc[comparison["model"] == "Dummy (majority)", "F1"].max() == 0.0
imbalanced_rows = comparison[comparison["dataset"] == "imbalanced"].set_index("model")
# On imbalanced data the dummy looks accurate but finds no positive case.
assert imbalanced_rows.loc["Dummy (majority)", "accuracy"] > 0.9
assert imbalanced_rows.loc["Dummy (majority)", "F1"] == 0.0
assert imbalanced_rows.loc["LogisticRegression", "F1"] > 0.0
print("Balanced and imbalanced comparison checks passed.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, dataset_name in zip(axes, ["balanced", "imbalanced"]):
    part = comparison[comparison["dataset"] == dataset_name]
    positions = np.arange(len(part))
    width = 0.35
    ax.bar(positions - width / 2, part["accuracy"], width, label="Accuracy", color="tab:blue")
    ax.bar(positions + width / 2, part["F1"], width, label="F1", color="tab:orange")
    ax.set_xticks(positions)
    ax.set_xticklabels(part["model"], rotation=15)
    ax.set_ylim(0, 1.05)
    ax.set_title(dataset_name.capitalize() + " dataset")
    ax.set_ylabel("Score")
    ax.legend()
plt.suptitle("Accuracy and F1: Dummy vs LogisticRegression")
plt.tight_layout()
plt.show()


**Reading the chart.** On the balanced data the dummy is only about as good as
guessing, while logistic regression scores much higher on both accuracy and F1.
On the imbalanced data the dummy has high accuracy near `0.95` but F1 exactly
`0`, because it predicts only the majority class. Logistic regression scores
higher on both, and its F1 shows that it actually finds positive cases. The exact
numbers depend on the random split; the pattern is the lesson: read F1 together
with accuracy when the classes are uneven.


## Training, validation and test performance

We divide the data into three parts:

- **Training set**: the model learns from it.
- **Validation set**: we use it to compare settings and pick one.
- **Test set**: we use it once at the end for the final score.

**Generalization** is how well a model does on data it has never seen. A model
that scores `0.99` on training data and `0.60` on unseen data has memorized the
training rows instead of learning the pattern. A model that scores about `0.85`
on both has learned something that transfers.

**Small numerical example.** Training accuracy `0.94`, validation `0.89`, test
`0.88`: the three scores are close, so the model generalizes. If training were
`0.99` and validation `0.60`, the gap would warn us about overfitting.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_balanced, y_balanced, test_size=0.4, random_state=RANDOM_STATE, stratify=y_balanced
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)
print("training rows:", len(X_train), "validation rows:", len(X_val), "test rows:", len(X_test))
assert len(X_train) + len(X_val) + len(X_test) == len(X_balanced)

split_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
split_model.fit(X_train, y_train)

split_scores = pd.DataFrame({
    "part": ["training", "validation", "test"],
    "accuracy": [
        accuracy_score(y_train, split_model.predict(X_train)),
        accuracy_score(y_val, split_model.predict(X_val)),
        accuracy_score(y_test, split_model.predict(X_test)),
    ],
})
print(split_scores.round(3))
assert split_scores["accuracy"].between(0, 1).all()


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(
    split_scores["part"],
    split_scores["accuracy"],
    color=["tab:blue", "tab:orange", "tab:green"],
)
plt.ylim(0, 1.05)
plt.title("Training, Validation and Test Accuracy")
plt.xlabel("Data part")
plt.ylabel("Accuracy")
plt.grid(axis="y")
plt.show()


**Reading the bar chart.** Training accuracy is the highest, then validation and
test are a little lower, which is the usual pattern: the model has already seen
the training rows. The three bars are close together, so the model generalizes
rather than memorizes. One split gives one estimate; the next section uses K-fold
cross-validation to average over several splits.


## K-fold cross-validation

**K-fold cross-validation** splits the data into `K` equal parts called folds.
It trains `K` times: each time one fold is the validation part and the other
`K - 1` folds are the training part. The final score is the average of the `K`
validation scores.

**Small numerical example (6 rows, 3 folds).**

```text
Fold 1: validate on rows 1-2, train on rows 3-6
Fold 2: validate on rows 3-4, train on rows 1, 2, 5, 6
Fold 3: validate on rows 5-6, train on rows 1-4
```

With two rows in each fold, a validation accuracy can only be `0`, `0.5` or
`1.0` (0, 1 or 2 correct out of 2). If the three folds score `0.5`, `1.0` and
`0.0`, the cross-validated accuracy is `(0.5 + 1.0 + 0.0) / 3 = 0.5`. The spread
of the three scores tells you how stable the model is.

**Stratified** K-fold keeps the class proportions in every fold, which matters
for imbalanced data. `StratifiedKFold` does this.

**Data leakage.** Preprocessing must be learned inside each fold, on that fold's
training part only. If you scale the whole dataset before cross-validation, the
scaler has already seen every validation fold, and the score can look too good.
Put the scaler and the model together in a `Pipeline`; then `cross_val_score`
fits the scaler again for each fold.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

# Cross-validate on the training split only, never the validation or test rows.
cv_accuracy = cross_val_score(cv_pipeline, X_train, y_train, cv=cv, scoring="accuracy")
cv_f1 = cross_val_score(cv_pipeline, X_train, y_train, cv=cv, scoring="f1")
print("training rows used for cross-validation:", len(X_train))
print("fold accuracy scores:", np.round(cv_accuracy, 3))
print("mean accuracy:", round(cv_accuracy.mean(), 3), "std:", round(cv_accuracy.std(), 3))
print("mean F1:", round(cv_f1.mean(), 3))

assert len(cv_accuracy) == 5
assert cv_accuracy.min() >= 0 and cv_accuracy.max() <= 1
assert np.isclose(cv_accuracy.mean(), cv_accuracy.sum() / 5)

# The scaler inside the pipeline is fitted again on each fold's training rows.
# Fitting one scaler on all training rows first would already use the rows that
# a fold should treat as unseen, so the statistics would be different.
train_index, val_index = next(iter(cv.split(X_train, y_train)))
scaler_fold_training = StandardScaler().fit(X_train[train_index])
scaler_all_training = StandardScaler().fit(X_train)
print("mean of feature 0 from fold-1 training rows:", round(scaler_fold_training.mean_[0], 3))
print("mean of feature 0 from all training rows:   ", round(scaler_all_training.mean_[0], 3))
assert not np.allclose(scaler_fold_training.mean_, scaler_all_training.mean_)
print("Cross-validation checks passed.")


**Reading the cross-validation output.** Cross-validation used only the 240
training rows; the validation and test rows set aside earlier stayed out. The
five fold scores (about `0.88` to `0.96`) are close, so the model is stable, and
their mean accuracy is about `0.93`, a fairer estimate than one split. The scaler
mean differs between fold 1's training rows and all training rows, which is
exactly why the scaler is fitted inside the pipeline: fitting it once on all rows
would let each fold use rows it should treat as unseen. A cross-validation score
is an estimate from training data, not a promise about future data.


## Underfitting, overfitting, bias and variance

A model has a **capacity**: how complex a pattern it can represent.

- **Underfitting** (high bias): the model is too simple. It scores poorly on both
  training and validation data.
- **Good fit**: the model matches the real pattern. Training and validation
  scores are close and good.
- **Overfitting** (high variance): the model is too complex. It fits the training
  noise and scores much worse on validation data.

**Bias** is the error from a too-simple assumption. **Variance** is how much the
model changes when the training sample changes. More capacity lowers bias and
raises variance; the best model balances them.

**Small numerical example.** Fit a curve to points that follow a gentle arc.

- A straight line (degree 1) is too stiff: training error `2.2`, validation
  error `2.4`. Underfitting.
- A curve of degree 3 follows the arc: training `0.4`, validation `0.9`. Good.
- A curve of degree 15 bends through every training point: training `0.2`, but
  validation `11.7`. Overfitting.

The dataset below is a small synthetic regression curve. It is used only to show
model capacity; it is not a real-world benchmark.


In [ ]:
regression_rng = np.random.default_rng(RANDOM_STATE)
x_all = np.linspace(-3, 3, 60).reshape(-1, 1)
# A curved pattern with random noise. Used only to show model capacity.
y_all = 0.5 * x_all.ravel() ** 2 - x_all.ravel() + 1 + regression_rng.normal(0, 1.0, size=60)

x_train, x_val, y_train_reg, y_val_reg = train_test_split(
    x_all, y_all, test_size=0.5, random_state=RANDOM_STATE
)
print("training points:", len(x_train), "validation points:", len(x_val))


In [ ]:
def fit_polynomial(degree):
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression()),
    ])
    model.fit(x_train, y_train_reg)
    train_error = mean_squared_error(y_train_reg, model.predict(x_train))
    val_error = mean_squared_error(y_val_reg, model.predict(x_val))
    return model, train_error, val_error


degrees = [1, 3, 15]
fitted_models = {}
rows = []
for degree in degrees:
    model, train_error, val_error = fit_polynomial(degree)
    fitted_models[degree] = model
    rows.append({"degree": degree, "train MSE": train_error, "validation MSE": val_error})

capacity_table = pd.DataFrame(rows)
print(capacity_table.round(3))

# Degree 1 underfits: validation error is higher than the good degree-3 fit.
assert capacity_table.loc[0, "validation MSE"] > capacity_table.loc[1, "validation MSE"]
# Degree 15 overfits: it fits training better but validation worse than degree 3.
assert capacity_table.loc[2, "train MSE"] < capacity_table.loc[1, "train MSE"]
assert capacity_table.loc[2, "validation MSE"] > capacity_table.loc[1, "validation MSE"]
print("Underfitting and overfitting checks passed.")


In [ ]:
x_line = np.linspace(-3, 3, 200).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, degree in zip(axes, degrees):
    ax.scatter(x_train, y_train_reg, color="tab:blue", label="Training points")
    ax.scatter(x_val, y_val_reg, color="tab:orange", alpha=0.6, label="Validation points")
    ax.plot(x_line, fitted_models[degree].predict(x_line), color="black",
            label="Degree " + str(degree) + " fit")
    ax.set_title("Degree " + str(degree))
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
plt.suptitle("Underfitting (1), Good Fit (3) and Overfitting (15)")
plt.tight_layout()
plt.show()


**Reading the three panels.** In the left panel the straight line cannot follow the
curve: it misses the pattern on both training and validation points
(underfitting). In the middle panel the degree-3 curve follows the arc and stays
close to the validation points (good fit). In the right panel the degree-15 curve
twists through the training points but swings wildly between them and misses the
validation points (overfitting). Compare the printed training and validation MSE
values: the overfit model wins on training and loses badly on validation.


## Regularization: L1 and L2

**Regularization** adds a penalty for large weights to the loss. This discourages
the model from chasing noise and usually improves validation performance.

### Small numerical example

Take one weight `w = 2` and penalty strength `alpha = 0.5`.

- L2 penalty: `alpha * w^2 = 0.5 * 2^2 = 2.0`.
- L1 penalty: `alpha * |w| = 0.5 * 2 = 1.0`.

The push on the weight also differs:

- L2 gradient: `2 * alpha * w = 2 * 0.5 * 2 = 2.0`, which grows as `w` grows.
- L1 gradient: `alpha * sign(w) = 0.5`, a constant push.

Now minimize a simple objective, `(w - 3)^2 + alpha * w^2`, with `alpha = 1`:

- Without a penalty the best `w` is `3`.
- L2: `2(w - 3) + 2w = 0`, so `w = 1.5`. The weight shrank from `3` to `1.5`.
- L1: `2(w - 3) + 1 = 0`, so `w = 2.5`. The weight shrank from `3` to `2.5`.

The important difference: L2 shrinks weights smoothly but rarely to exactly zero.
L1 applies a constant push, so weak weights can land exactly on zero. This makes
L1 useful for feature selection.

General forms, with weights `w_j` and strength `alpha` (also called `lambda`):

```text
total cost = prediction error + alpha * sum of w_j^2      (L2, Ridge)
total cost = prediction error + alpha * sum of |w_j|      (L1, Lasso)
```

Here the sum runs over all weights. A larger `alpha` means a stronger penalty and
smaller weights. The bias term is usually not penalized.


In [ ]:
def fit_regularized(regressor):
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=15, include_bias=False)),
        ("scaler", StandardScaler()),
        ("regressor", regressor),
    ])
    model.fit(x_train, y_train_reg)
    train_error = mean_squared_error(y_train_reg, model.predict(x_train))
    val_error = mean_squared_error(y_val_reg, model.predict(x_val))
    coefficients = model.named_steps["regressor"].coef_
    zero_count = int(np.sum(np.isclose(coefficients, 0, atol=1e-8)))
    return model, train_error, val_error, coefficients, zero_count


regressors = {
    "No penalty (LinearRegression)": LinearRegression(),
    "L2 (Ridge, alpha=1.0)": Ridge(alpha=1.0),
    "L1 (Lasso, alpha=0.01)": Lasso(alpha=0.01, max_iter=50000),
}

regularization_rows = []
coefficients = {}
for name, regressor in regressors.items():
    model, train_error, val_error, coef, zero_count = fit_regularized(regressor)
    coefficients[name] = coef
    regularization_rows.append({
        "model": name,
        "train MSE": train_error,
        "validation MSE": val_error,
        "zero weights": zero_count,
    })

regularization_table = pd.DataFrame(regularization_rows)
print(regularization_table.round(3))

# Ridge shrinks weights but keeps them; Lasso sets some to exactly zero.
assert regularization_table.loc[0, "train MSE"] <= regularization_table.loc[1, "train MSE"]
assert regularization_table.loc[1, "validation MSE"] < regularization_table.loc[0, "validation MSE"]
assert regularization_table.loc[2, "zero weights"] > regularization_table.loc[1, "zero weights"]
print("Regularization checks passed.")


In [ ]:
feature_numbers = np.arange(len(coefficients["L2 (Ridge, alpha=1.0)"]))
width = 0.4
plt.figure(figsize=(12, 4))
plt.bar(feature_numbers - width / 2,
        np.abs(coefficients["L2 (Ridge, alpha=1.0)"]), width,
        label="L2 Ridge", color="tab:blue")
plt.bar(feature_numbers + width / 2,
        np.abs(coefficients["L1 (Lasso, alpha=0.01)"]), width,
        label="L1 Lasso", color="tab:red")
plt.title("Coefficient Sizes: L2 Shrinks, L1 Sets Some to Zero")
plt.xlabel("Polynomial feature number")
plt.ylabel("Absolute coefficient value")
plt.legend()
plt.grid(axis="y")
plt.show()


**Reading the table and the chart.** Without a penalty the model has the lowest
training error (`0.242`) but the worst validation error (`11.744`): it overfits.
Ridge (L2) trades a little training error for a much better validation error
(`1.050`) and keeps all 15 weights, only smaller. Lasso (L1) sets 9 of the 15
weights exactly to zero and reaches validation error about `1.002`, so it is both
regularized and simpler. The chart shows the same idea: L1 has many bars at zero,
while L2 has small but non-zero bars everywhere. The exact numbers depend on the
data and the chosen `alpha`.


## Hyperparameters and hyperparameter tuning

Two kinds of numbers control a model:

- **Learned parameters**: found by fitting the data, such as weights and the bias
  in linear regression, or the coefficients in a polynomial.
- **Hyperparameters**: chosen before fitting, such as the polynomial degree, the
  regularization strength `alpha`, or the `C` value in logistic regression.

**Tuning** means trying several hyperparameter values and keeping the best one.
The search must use training data only, usually with cross-validation on the
training part. The test set stays untouched until the end, when we report one
final score.

**Small numerical example.** Suppose we try `C` values `[0.01, 0.1, 1, 10]` with
5-fold cross-validation on the training part. If the mean validation scores are
`[0.71, 0.85, 0.91, 0.93]`, the largest score is `0.93`, so we keep the `C` value
that produced it. Then we evaluate that chosen model once on the test set and
report that single number.


In [ ]:
X_tune, y_tune = make_classification(
    n_samples=300,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.5, 0.5],
    class_sep=1.5,
    flip_y=0.02,
    random_state=7,
)
X_tune_train, X_tune_test, y_tune_train, y_tune_test = train_test_split(
    X_tune, y_tune, test_size=0.25, random_state=RANDOM_STATE, stratify=y_tune
)

tuning_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
# C is the inverse regularization strength: small C means strong regularization.
param_grid = {"model__C": [0.01, 0.1, 1.0, 10.0]}
grid_search = GridSearchCV(
    tuning_pipeline,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1",
)
grid_search.fit(X_tune_train, y_tune_train)
print("best hyperparameters:", grid_search.best_params_)
print("best cross-validation F1 (training data only):", round(grid_search.best_score_, 3))

# The test set is used once, after tuning, for the final honest score.
final_predictions = grid_search.predict(X_tune_test)
final_accuracy = accuracy_score(y_tune_test, final_predictions)
final_f1 = f1_score(y_tune_test, final_predictions, pos_label=1, zero_division=0)
print("final test accuracy:", round(final_accuracy, 3))
print("final test F1:", round(final_f1, 3))

assert grid_search.best_params_["model__C"] in param_grid["model__C"]
assert len(grid_search.cv_results_["mean_test_score"]) == len(param_grid["model__C"])
assert 0 <= final_accuracy <= 1 and 0 <= final_f1 <= 1
print("Hyperparameter tuning checks passed.")


**Reading the tuning output.** The best `C` and its cross-validation F1 come from
the training data only; the test rows were not used to choose `C`. The final test
accuracy and F1 are then computed once. A cross-validation score is an estimate,
not a guarantee, and a single test score also has uncertainty. Use the test
result as an honest final report, not as a reason to keep tuning.


## Common mistakes

- **Forgetting the positive class.** Precision, recall, F1 and Jaccard depend on
  which label is positive. Always state it.
- **Reading accuracy alone on imbalanced data.** A majority-only model can look
  accurate but has F1 `0`.
- **Swapping FP and FN.** FP is a false alarm; FN is a missed case. They have
  different costs in real problems.
- **Feeding labels to log loss.** Log loss needs probabilities, not hard `0`/`1`
  predictions.
- **Scaling before splitting or before cross-validation.** This leaks
  information. Put the scaler in a `Pipeline`.
- **Using the test set many times.** Each peek makes the final score less honest.
  Tune on validation or cross-validation; test once.
- **Judging fit from training error only.** A very low training error can mean
  overfitting. Compare training and validation scores.
- **Making the model more complex for every problem.** More capacity lowers bias
  and raises variance; regularization and simpler models often generalize better.
- **Reporting a single split as the truth.** Cross-validation gives a more stable
  estimate.


## Summary

In this notebook we measured and improved models:

- The **confusion matrix** holds TN, FP, FN and TP for a stated positive class.
- **Accuracy, precision, recall, F1 and Jaccard** each answer a different
  question; F1 and Jaccard matter most on imbalanced data.
- **Log loss** scores probabilities and punishes confident wrong predictions.
- A **majority dummy** can have high accuracy and F1 `0` on imbalanced data.
- **Training, validation and test** sets separate learning, choosing and final
  reporting; **generalization** means doing well on unseen data.
- **K-fold cross-validation** averages several splits; `StratifiedKFold` keeps
  class proportions, and a `Pipeline` keeps preprocessing inside each fold to
  avoid **data leakage**.
- **Underfitting** (high bias), **good fit** and **overfitting** (high variance)
  show up as training and validation scores that are both poor, close, or far
  apart.
- **Regularization** adds a penalty: **L2 (Ridge)** shrinks weights smoothly,
  **L1 (Lasso)** can set them exactly to zero.
- **Hyperparameters** are chosen before fitting; tune them on training data with
  `GridSearchCV`, then evaluate the test set once.
